# 第 15 章 IS-MP-PC モデル: 数字を変えて確かめる

**使い方**: 「Parameters」のセルの数字を書き換えて、上から順に実行します（メニューの Run → Run All Cells）。数字を変えるたびに、結果と図が変わります。

起動: アプリのディレクトリで `uv run --with jupyterlab jupyter lab notebooks/` を実行します。

記事: `docs/article/macro-programming/15-is-mp-pc.md`

In [ ]:
# --- Parameters: ここの数字を書き換えます ---
a = 1.0         # IS 曲線の傾き
b = 0.25        # フィリップス曲線の傾き
c = 1.5         # MP 曲線のインフレ率への反応
d = 0.5         # MP 曲線の産出量への反応
shock_pc = 1.0  # 1 期間だけの、フィリップス曲線へのショック
shock_mp = 0.0  # 1 期間だけの、金融政策のショック

In [ ]:
# --- 準備（変更しません）: lib を読み込めるようにします ---
import sys
from pathlib import Path

_root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "lib").is_dir())
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

## ショックへの反応

In [ ]:
from lib.chapter15.ismppc import IsMpPcParameters, impulse_response

params = IsMpPcParameters(a=a, b=b, c=c, d=d)
df = impulse_response(params, shock_pc=shock_pc, shock_mp=shock_mp)
shock = df.iloc[3]
after = df.iloc[4]
pi_star = params.pi_star
print(f"ショックの期: インフレ率 {shock['pi']:.4f}、産出量 {shock['y']:.4f}")
print(f"次の期のインフレ率の乖離: {after['pi'] - pi_star:.4f}")
print(f"φ = {params.phi:.4f}（乖離は、毎期 φ 倍になる）")

## 図: ショックへの反応

In [ ]:
from lib.chapter15.plot import plot_responses

plot_responses(df)

## 図: 金融政策の姿勢の比較（基本と、産出量を重視）

In [ ]:
from lib.chapter15.plot import plot_stance_comparison

plot_stance_comparison(
    impulse_response(IsMpPcParameters(a=a, b=b), shock_pc=shock_pc),
    df,
)

## 試してみよう

- `c = 0.5`、`d = 1.5` にすると、産出量の落ち込みは小さく（99.8095）なりますが、
  インフレ率は 2.9524 に上がり、φ は 0.9524 に大きくなります（乖離が長く続きます）
- `shock_pc = 0.0`、`shock_mp = 1.0` にすると、金融政策のショックで、
  インフレ率は 1.8667 に、産出量は 99.4667 に下がります
- `c = 3.0` にすると、φ は 0.6667 に下がり、
  次の期のインフレ率の乖離は 0.4444 です（インフレ率に強く反応するほど、
  乖離は早く縮みます）